In [ ]:
!pip install langchain langchain-community pypdf pymupdf pyvis networkx seaborn google-generativeai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Imports

In [ ]:
import pandas as pd
import numpy as np
import os, uuid, json, random
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import google.generativeai as genai
import networkx as nx
from pyvis.network import Network
import seaborn as sns

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
GEMINI_API_KEY = "API-KEY"
PDF_PATH = "/content/inputs/text-input.pdf"

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")

Load/Chunk PDF

In [ ]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
pages = splitter.split_documents(documents)
print(f"{len(pages)} chunks loaded")

289 chunks loaded


Build Chunk DataFrame

In [ ]:
def documents2Dataframe(documents):
    return pd.DataFrame([
        {"text": c.page_content, **c.metadata, "chunk_id": uuid.uuid4().hex}
        for c in documents
    ])

Concept Extraction

In [ ]:
def extractConcepts(text: str, metadata: dict) -> list:
    SYS = """Extract key concepts from the context. Return ONLY a valid JSON array:
[{"entity": "concept name", "importance": 1-5, "category": "event|concept|place|object|document|organisation|condition|misc"}, ...]
No explanation, no markdown, just JSON. Ensure all strings are properly escaped."""
    try:
        response = model.generate_content(f"{SYS}\n\nContext: ```{text}```")
        raw = response.text.strip()

        # Strip markdown code fences if present
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

        # Extract just the JSON array (find first '[' to last ']')
        start = raw.find("[")
        end = raw.rfind("]") + 1
        if start == -1 or end == 0:
            print(f"No JSON array found in response: {raw[:200]}")
            return []
        raw = raw[start:end]

        # Fix common issues: replace smart quotes, strip control characters
        raw = raw.replace("\u2018", "'").replace("\u2019", "'")
        raw = raw.replace("\u201c", '"').replace("\u201d", '"')
        raw = "".join(c for c in raw if ord(c) >= 32 or c in "\n\t")

        result = json.loads(raw)
        return [dict(item, **metadata) for item in result]

    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        # Last resort: try to salvage complete objects before the error position
        try:
            partial = raw[:e.pos].rstrip().rstrip(",") + "]"
            result = json.loads(partial)
            print(f"   ↳ Salvaged {len(result)} concepts from partial response")
            return [dict(item, **metadata) for item in result]
        except Exception:
            print(f"   ↳ Could not salvage. Raw snippet: {raw[max(0,e.pos-100):e.pos+50]}")
            return []
    except Exception as e:
        print(f"Skipped concept extraction: {e}")
        return []

def df2ConceptsList(dataframe: pd.DataFrame) -> list:
    """Run extractConcepts over every row and flatten into one list."""
    all_concepts = []
    for _, row in dataframe.iterrows():
        concepts = extractConcepts(row["text"], {"chunk_id": row["chunk_id"], "type": "concept"})
        all_concepts.extend(concepts)
    return all_concepts

def concepts2Df(concepts_list: list) -> pd.DataFrame:
    """Clean and normalise the raw concepts list into a deduplicated dataframe."""
    df_concepts = pd.DataFrame(concepts_list).replace(" ", np.nan)
    df_concepts = df_concepts.dropna(subset=["entity"])
    df_concepts["entity"] = df_concepts["entity"].str.lower()
    return df_concepts

Graph Extraction

In [ ]:
def graphPrompt(text: str, metadata: dict) -> list:
    SYS = """You are a network graph maker. Extract terms and their relations from the context.
Return ONLY a valid JSON array:
[{"node_1": "concept A", "node_2": "concept B", "edge": "relationship in one or two sentences"}, ...]
No explanation, no markdown, just JSON."""
    try:
        response = model.generate_content(f"{SYS}\n\nContext: ```{text}```\n\nOutput:")
        raw = response.text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return [dict(item, **metadata) for item in json.loads(raw)]
    except Exception as e:
        print(f"Skipped graph extraction: {e}")
        return []

def df2Graph(dataframe: pd.DataFrame) -> list:
    """Run graphPrompt over every row and flatten into one list."""
    all_edges = []
    for _, row in dataframe.iterrows():
        edges = graphPrompt(row["text"], {"chunk_id": row["chunk_id"]})
        all_edges.extend(edges)
    return all_edges

def graph2Df(nodes_list: list) -> pd.DataFrame:
    """Clean and normalise the raw edges list into a dataframe."""
    dfg = pd.DataFrame(nodes_list).replace("", np.nan).dropna(subset=["node_1", "node_2", "edge"])
    dfg["node_1"] = dfg["node_1"].str.lower()
    dfg["node_2"] = dfg["node_2"].str.lower()
    return dfg

Contextual Proximity

In [ ]:
def contextual_proximity(df: pd.DataFrame) -> pd.DataFrame:
    long = pd.melt(df, id_vars=["chunk_id"], value_vars=["node_1", "node_2"], value_name="node").drop(columns=["variable"])
    wide = pd.merge(long, long, on="chunk_id", suffixes=("_1", "_2"))
    wide = wide[wide["node_1"] != wide["node_2"]].reset_index(drop=True)
    wide = wide.groupby(["node_1", "node_2"]).agg({"chunk_id": [",".join, "count"]}).reset_index()
    wide.columns = ["node_1", "node_2", "chunk_id", "count"]
    wide = wide[wide["count"] > 1]
    wide["edge"] = "contextual proximity"
    return wide

Pipeline

In [ ]:
df = documents2Dataframe(pages)
print(df.shape)

os.makedirs("/content/outputs", exist_ok=True)
regenerate = True

if regenerate:
    print("🔍 Extracting concepts...")
    concepts_list = df2ConceptsList(df)
    df_concepts = concepts2Df(concepts_list)
    df_concepts.to_csv("/content/outputs/concepts.csv", index=False)
    print(f"{len(df_concepts)} concepts extracted")
    print(df_concepts.head())

    print("\n🔗 Extracting graph edges...")
    edges_list = df2Graph(df)
    dfg1 = graph2Df(edges_list)
    dfg1["count"] = 4
    dfg1.to_csv("/content/outputs/graph.csv", sep="|", index=False)
    df.to_csv("/content/outputs/chunks.csv", sep="|", index=False)
    print(f"{len(dfg1)} edges extracted")
else:
    dfg1 = pd.read_csv("/content/outputs/graph.csv", sep="|")
    df_concepts = pd.read_csv("/content/outputs/concepts.csv")

dfg1.replace("", np.nan, inplace=True)
dfg1.dropna(subset=["node_1", "node_2", "edge"], inplace=True)

(289, 9)
🔍 Extracting concepts...


Merge direct/contextual proximity edges

In [ ]:
dfg2 = contextual_proximity(dfg1)
dfg = pd.concat([dfg1, dfg2]).groupby(["node_1", "node_2"]).agg(
    {"chunk_id": ",".join, "edge": ",".join, "count": "sum"}
).reset_index()

NetworkX Graph

In [ ]:
nodes = pd.concat([dfg["node_1"], dfg["node_2"]], axis=0).unique()
print(f"\n{len(nodes)} nodes")

G = nx.Graph()
for node in nodes:
    G.add_node(str(node))
for _, row in dfg.iterrows():
    G.add_edge(str(row["node_1"]), str(row["node_2"]), title=row["edge"], weight=row["count"] / 4)

Community Detection (depth=2)

In [ ]:
communities_generator = nx.community.girvan_newman(G)
top_level_communities = next(communities_generator)
next_level_communities = next(communities_generator)
communities = sorted(map(sorted, next_level_communities))
print(f"{len(communities)} communities found")

Render Graph with Pyvis

In [ ]:
palette = sns.color_palette("hls", len(communities)).as_hex()
random.shuffle(palette)

for i, community in enumerate(communities):
    color = palette[i]
    for node in community:
        if node in G.nodes:
            G.nodes[node]["color"] = color
            G.nodes[node]["group"] = i
            G.nodes[node]["size"] = G.degree[node]

In [ ]:
net = Network(height="900px", width="100%", cdn_resources="remote", select_menu=True)
net.from_nx(G)
net.force_atlas_2based(central_gravity=0.015, gravity=-31)
net.show_buttons(filter_=["physics"])
net.show("graph.html", notebook=False)

from IPython.display import IFrame
IFrame("graph.html", width="100%", height=900)

from google.colab import files
files.download("graph.html")